# 03 — Model evaluation (Person 3)

Customer-relative deviation features → LR baseline vs Random Forest.

**Do not headline accuracy.** Report Precision, Recall, F1, PR-AUC.

In [ ]:
import json
from pathlib import Path
import sys

sys.path.append("..")

import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay, ConfusionMatrixDisplay

from src.config import DATA_FEATURES, FEATURE_COLS, METRICS_PATH, RF_PATH, TARGET, TIME_COL
from src.explain import score_one, try_tree_explainer
from src.train import compute_metrics, time_split

In [ ]:
df = pd.read_csv(DATA_FEATURES, parse_dates=[TIME_COL])
print("rows", len(df), "fraud rate", f"{df[TARGET].mean():.2%}")
df.head()

In [ ]:
train, test = time_split(df)
print("train", len(train), "test", len(test))
print("Do not randomly shuffle when timestamps exist — later txns are the test set.")

In [ ]:
metrics = json.loads(Path(METRICS_PATH).read_text())
pd.DataFrame(
    {
        "logistic_regression": metrics["logistic_regression"],
        "random_forest": metrics["random_forest"],
    }
).T[["precision", "recall", "f1", "pr_auc", "accuracy"]]

In [ ]:
rf = joblib.load(RF_PATH)
X_test, y_test = test[FEATURE_COLS], test[TARGET]
prob = rf.predict_proba(X_test)[:, 1]
print(compute_metrics(y_test, prob))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
PrecisionRecallDisplay.from_predictions(y_test, prob, ax=ax[0], name="RF")
ConfusionMatrixDisplay.from_predictions(y_test, (prob >= 0.5).astype(int), ax=ax[1])
ax[0].set_title("PR curve")
plt.tight_layout()

In [ ]:
imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values()
imp.plot(kind="barh", title="Global RF feature importance")

In [ ]:
idx = int(prob.argmax())
row = X_test.iloc[[idx]]
explainer = try_tree_explainer(rf)
print("highest-risk test row")
display(test.iloc[[idx]])
print(score_one(row, rf, explainer))